# 04 - Local vs external pollution, and what sensors can (and cannot) tell apart

*Idealised synthetic experiment. Every number is a demonstration value from a configuration file, not an estimate for any real place.*

Notebooks 01-03 had one kind of source. Here we split the pollution that reaches a **study region** by where it came from, and ask a hard question: *if we only have noisy sensor readings of the total concentration, can we recover how much came from each origin?*

**Terms used literally in this notebook**
1. **Local**: emitted *inside* the study region (an explicit source whose centre lies in it).
2. **External**: originating *outside* the study region: either an explicit source placed outside it, or a concentration entering through the outer inflow boundary of the simulation domain (a "boundary basis", standing for distant sources we do not resolve).
3. **International transboundary**: pollution crossing a national border. We do **not** use this word for the synthetic domain, because no verified national boundary exists there; the study region is just a square used to report results.
4. **Indoor**: air inside a building that exchanges with the outdoors; treated in a separate section at the end, one-way coupled to the outdoor field.
5. **The study region is a reporting mask, not a wall.** The PDE is solved on the larger simulation domain; the region only decides which cells count as "inside".

**Bounded run.** This notebook uses the *smoke* configuration (`configs/transboundary_smoke.json`): domain A is 30 x 30 cells of 500 m and the run lasts 7200 s, so that the whole notebook runs in a few minutes. Unit responses are computed once and cached. The research profile (finer grid, 6 h, ensembles, validation-selected regularisation) is `configs/transboundary.json`, run from the command line; see `docs/transboundary.md`.

## 0. Setup and parameters

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, FancyArrowPatch
from pathlib import Path

from pravah.config import load_config, SolverOptions, Initial
from pravah import transboundary as tb
from pravah.attribution import (contributions, component_errors, emission_error, external_part, total_field,
                                origin_fractions, identifiability_report)
from pravah.inverse import ResponseCache, InferenceOptions, fit_sources
from pravah.observations import SensorLayout, ObservationSpec, observe, sample
from pravah.experiments import perturb_wind
from pravah import indoor
%matplotlib inline
T_START = time.perf_counter()
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [ ]:
# ---------------- PARAMETERS (edit these) ----------------
PROFILE = "transboundary_smoke.json"   # bounded profile; use transboundary.json from the command line for research runs
NOISE_FRACTION = 0.02       # noise std as a fraction of the stated signal scale (2 % = quiet, 10 % = noisy)
NOISE_FRACTION_HIGH = 0.10  # the second noise level used for comparisons
ALPHA_MODERATE = 1.0        # moderate ridge strength (scaled units; NOT tuned on this notebook's own data)
ALPHA_STRONG = 1e5          # strong prior: the estimate is pulled to the prior (zero)
N_REPS = 8                  # noise repetitions for the layout comparison
SEED = 1                    # noise seed of the single illustrative case
# ---------------------------------------------------------

root = Path.cwd()
while not (root / "configs" / PROFILE).exists() and root != root.parent:
    root = root.parent
cfg = load_config(root / "configs" / PROFILE)
solver = SolverOptions()
cache = ResponseCache()            # every unit response is computed once and reused
T_EXCL = float(cfg["physics"]["t_exclude"])
BUF = float(cfg["geometry"]["domains"]["A"]["buffer"])
STUDY = float(cfg["geometry"]["study_size"])
T_END = float(cfg["physics"]["T"])

def obs_times(T):
    return tuple(float(cfg["recovery"]["obs_interval"]) * (i + 1) for i in range(int(round(T / cfg["recovery"]["obs_interval"]))))

def to_layout(points, prefix="S"):
    # config coordinates are STUDY-RELATIVE metres; the simulation domain adds a buffer BUF on every side
    return SensorLayout(tuple(f"{prefix}{i + 1}" for i in range(len(points))),
                        tuple(float(p[0]) + BUF for p in points), tuple(float(p[1]) + BUF for p in points))

LAYOUTS = {k: to_layout(v) for k, v in cfg["recovery"]["layouts"].items()}
OBS_TIMES = obs_times(T_END)

def zero_parts(resp):
    # all scenarios here have zero initial field and zero static background, so those two tags are identically zero
    z = np.zeros_like(resp.base)
    return {"initial": z, "background": z}

scn, theta, labels = tb.build_scenario(cfg, "mixed")
assert scn.initial == Initial() and scn.boundary.background == 0.0
print(f"domain A: {scn.domain.nx} x {scn.domain.ny} cells, dx = {scn.domain.dx:.0f} m, T = {T_END:.0f} s, "
      f"study region = {STUDY:.0f} m square, evaluation window [{T_EXCL:.0f}, {T_END:.0f}] s")

## 1. The picture: study region, simulation domain, inflow boundary, sources, wind, sensors

The grey outer square is the **simulation domain** (where the equations are solved). The black square is the **study region**. The four coloured edges are the **outer inflow boundary sectors**: wherever the wind blows *into* the domain through them, a boundary-basis concentration (µg/m³) can enter. Circles are Gaussian sources (radius = 1 sigma); **local** sources lie inside the black square, **external** ones outside. Triangles are sensors; the two outside the study region (but inside the domain) are the *upwind / background reference* sensors used in section 6.

In [ ]:
def draw_setup(ax, scn, theta, labels, wind_deg, layouts):
    d = scn.domain
    ax.add_patch(Rectangle((0, 0), d.lx, d.ly, fill=False, ec="0.5", lw=1.5))
    st = scn.study
    ax.add_patch(Rectangle((st.x0, st.y0), st.x1 - st.x0, st.y1 - st.y0, fill=False, ec="k", lw=2))
    ax.text(st.x0 + 150, st.y1 - 450, "study region", fontsize=9)
    cols = {"west": "tab:blue", "south": "tab:green", "east": "tab:orange", "north": "tab:purple"}
    pad = 120
    ax.plot([0, 0], [0, d.ly], c=cols["west"], lw=4, label="west sector")
    ax.plot([0, d.lx], [0, 0], c=cols["south"], lw=4, label="south sector")
    ax.plot([d.lx, d.lx], [0, d.ly], c=cols["east"], lw=4, label="east sector")
    ax.plot([0, d.lx], [d.ly, d.ly], c=cols["north"], lw=4, label="north sector")
    for s in scn.sources:
        loc = s.group == "local"
        ax.add_patch(Circle((s.x, s.y), s.sigma, color="tab:red" if loc else "tab:brown", alpha=0.5))
        ax.text(s.x + 200, s.y + 200, f"{s.name} ({s.group})", fontsize=8)
    th = np.radians(wind_deg)
    ax.add_patch(FancyArrowPatch((st.x0 + 600, st.y0 - 1400), (st.x0 + 600 + 2200 * np.cos(th), st.y0 - 1400 + 2200 * np.sin(th)),
                                 arrowstyle="-|>", mutation_scale=18, color="k"))
    ax.text(st.x0 + 600, st.y0 - 2200, "wind (towards)", fontsize=8)
    lay = layouts["interior+upwind"]
    inside = [i for i in range(len(lay)) if st.x0 <= lay.x[i] <= st.x1 and st.y0 <= lay.y[i] <= st.y1]
    outside = [i for i in range(len(lay)) if i not in inside]
    ax.scatter([lay.x[i] for i in inside], [lay.y[i] for i in inside], marker="^", c="tab:blue", s=45, label="interior sensors", zorder=5)
    ax.scatter([lay.x[i] for i in outside], [lay.y[i] for i in outside], marker="v", c="tab:green", s=60, label="upwind reference sensors", zorder=5)
    ax.set_xlim(-600, d.lx + 600); ax.set_ylim(-600, d.ly + 600); ax.set_aspect("equal")
    ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")

fig, ax = plt.subplots(figsize=(7, 7))
draw_setup(ax, scn, theta, labels, cfg["wind"]["toward_deg"], LAYOUTS)
ax.legend(fontsize=7, loc="upper left", ncol=2)
ax.set_title("Simulation domain, study region, inflow sectors, sources, wind, sensors")
plt.show()
print(pd.DataFrame(labels)[["name", "group", "kind", "units", "true_value"]].to_string(index=False))

The parameter table above shows the **unknowns**: emission amplitudes (units µg/s, one per explicit source) and a boundary-basis amplitude (units µg/m³). These two kinds have different units and are **never compared by raw magnitude**; errors are always reported per kind.

**One rule we enforce:** the explicit source `E1` and the uniform boundary basis in the `mixed` scenario represent *different* things (E1 is one resolved plume; the basis stands for other, distant sources beyond the domain). The code refuses to let a basis stand for a plume that is also an explicit source (`tb.check_no_double_count`); otherwise the same plume would be counted twice.

## 2. Where does the pollution come from? Contribution fields

The model is **linear** in the amplitudes, so the total field is the sum of one field per origin tag (superposition). The tag follows the *origin of the mass*, not where it currently is. Below, for the `mixed` scenario at the final time, we show the local part, the external explicit-source part, the boundary (background) part, and the total, all on **one common colour scale**.

In [ ]:
resp_mixed = cache.get(scn, solver)
contrib = contributions(resp_mixed, theta, zero_parts(resp_mixed))
total = total_field(contrib)
direct = resp_mixed.field(theta)
print("max |sum of tags - total field| (ug/m3):", float(abs(total - direct).max()))

def show_fields(panels, title, t=T_END, ncols=4):
    vmax = max(float(da.sel(time=t).max()) for _, da in panels)
    fig, axes = plt.subplots(1, len(panels), figsize=(3.6 * len(panels), 3.4), squeeze=False)
    d = scn.domain
    for ax, (name, da) in zip(axes[0], panels):
        im = ax.imshow(da.sel(time=t).values, origin="lower", extent=(0, d.lx, 0, d.ly), vmin=0, vmax=vmax, cmap="viridis")
        ax.add_patch(Rectangle((scn.study.x0, scn.study.y0), STUDY, STUDY, fill=False, ec="w", lw=1.2))
        ax.set_title(name, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(im, ax=list(axes[0]), shrink=0.8, label="ug/m3 (common scale)")
    fig.suptitle(title, y=1.02)
    plt.show()

show_fields([("local only", contrib["local"]), ("external (explicit E1) only", contrib["external_sources"]),
             ("boundary (background) only", contrib["boundary"]), ("mixed = total", total)],
            f"mixed scenario at t = {T_END:.0f} s (white square = study region)")

In [ ]:
rows = []
for nm in ("local_only", "external_sources", "boundary_only", "mixed"):
    s, th_, _ = tb.build_scenario(cfg, nm)
    r = cache.get(s, solver)
    c = contributions(r, th_, zero_parts(r))
    of = origin_fractions(c, s, T_EXCL)
    rows.append({"scenario": nm, "n_params": s.n_params, "local fraction": of["local"], "external fraction": of["external"],
                 "peak total in study (ug/m3)": float(total_field(c).sel(time=slice(T_EXCL, None)).max())})
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Definition of the external fraction (a concentration quantity, not a mass flux).**
`F_ext = sum_w(external) / sum_w(total)`, where `sum_w` adds over the study-region cells (weights = the fraction of each cell's area inside the region) multiplied by trapezoid time weights over the output times in `[t_exclude, T]`. "External" = explicit external sources + boundary basis + static background. It is **undefined** (reported as `None`) when the total is essentially zero. The fraction of *mass crossing the boundary* is a different quantity and is not substituted. Check that the undefined case really returns `None`:

In [ ]:
zero_c = contributions(resp_mixed, np.zeros_like(theta), zero_parts(resp_mixed))
print("F_ext with all emissions zero:", origin_fractions(zero_c, scn, T_EXCL)["external"])
print("F_ext for the mixed scenario :", round(origin_fractions(contrib, scn, T_EXCL)["external"], 4))

## 3. Plume arrival delays and wind-direction dependence

Pollution from outside does not appear instantly: it must be carried in. `tb.arrival_time` reports the first time the *external* concentration at a point exceeds a small fraction (the config's `arrival_threshold`) of its own maximum, interpolating between output times; it returns `None` if the trace is numerically zero (no arrival).

In [ ]:
fl = cfg["fields"]
times = np.asarray(contrib["local"]["time"].values, dtype=float)
ext = external_part(contrib)
floor = 1e-3 * float(total.max())
fig, ax = plt.subplots(figsize=(6, 3.4))
rows = []
for k, (px, py) in enumerate(fl["arrival_points"]):
    lay = SensorLayout((f"P{k}",), (px + BUF,), (py + BUF,))
    tr = np.asarray(sample(ext, lay, times).values[:, 0], dtype=float)
    ta = tb.arrival_time(tr, times, float(fl["arrival_threshold"]), floor)
    rows.append({"point (study-relative m)": (px, py), "arrival time (s)": ta, "max external (ug/m3)": float(tr.max())})
    ax.plot(times, tr, label=f"P{k} ({px:.0f}, {py:.0f})")
    if ta is not None:
        ax.axvline(ta, ls=":", c=ax.lines[-1].get_color())
ax.set_xlabel("time (s)"); ax.set_ylabel("external concentration (ug/m3)"); ax.legend(fontsize=8)
ax.set_title("External concentration at points; dotted = arrival time")
plt.show()
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Wind-direction dependence.** We keep every source fixed, rotate only the wind (direction the air moves *towards*, counter-clockwise from east, 0 = east), and recompute the study-region external fraction `F_ext` with the weighting above (area fractions x trapezoid time weights over `[t_exclude, T]`). Each direction needs its own forward solves (a few seconds each).

In [ ]:
n_dir = 8
sweep = []
for nm in ("external_sources", "mixed"):
    for k in range(n_dir):
        deg = 360.0 * k / n_dir
        s, th_, _ = tb.build_scenario(cfg, nm, "A", tb.uniform_wind(float(cfg["wind"]["speed"]), deg))
        r = ResponseCache().get(s, solver)
        f = origin_fractions(contributions(r, th_, zero_parts(r)), s, T_EXCL)["external"]
        sweep.append({"scenario": nm, "toward_deg": deg, "F_ext": np.nan if f is None else f})
sweep = pd.DataFrame(sweep)
fig, ax = plt.subplots(figsize=(6, 3.4))
for nm, g in sweep.groupby("scenario"):
    ax.plot(g["toward_deg"], g["F_ext"], "o-", label=nm)
ax.set_xlabel("wind towards (deg, 0 = east)"); ax.set_ylabel("concentration-derived external fraction")
ax.set_xticks(range(0, 360, 45)); ax.legend(); ax.set_title("External fraction vs wind direction (gaps = undefined)")
plt.show()
print(sweep.pivot(index="toward_deg", columns="scenario", values="F_ext").round(4).to_string())

Reading it: E1 sits **west** of the study region, so the external-source fraction is largest when the wind carries air towards the east (0 degrees) and falls towards zero when the wind blows away from the region. The `mixed` curve is higher at the "wrong" directions because the boundary basis enters through whichever sides are upwind. A single external fraction therefore depends on the wind that happened to blow, and is never a fixed property of "the place".

## 4. Recovering the origin of pollution from TOTAL noisy observations

Now the inverse problem. The sensors see only the **total** concentration plus noise. We assume everything except the amplitudes is known (wind, `K`, decay, source positions and widths, basis shape), as in notebook 02, and fit the four amplitudes `L1, L2, E1` (µg/s) and `uniform` (µg/m³) with a non-negative least squares fit. The origin tags are used **only afterwards**, to score the answer; the fit never sees them.

**How the noise is set.** Noise is a *fraction of a stated signal scale*: the largest noiseless reading of the interior layout over the whole run. We print it in µg/m³ and in percent of that scale, so a reader can judge it.

In [ ]:
def signal_scale(truth_resp, truth_theta, layout, times):
    return float(sample(truth_resp.field(truth_theta), layout, times).values.max())

SCALE = signal_scale(resp_mixed, theta, LAYOUTS["interior"], OBS_TIMES)   # fixed stated signal scale (mixed, interior layout)
peak_study = float(total.sel(time=slice(T_EXCL, None)).max())
def noise_report(frac, scale=SCALE):
    sig = frac * scale
    return f"noise std = {sig:.5f} ug/m3 = {100 * frac:.1f} % of the signal scale ({scale:.4f} ug/m3) = {100 * sig / peak_study:.1f} % of the peak study-region total ({peak_study:.4f} ug/m3)"
for f in (NOISE_FRACTION, NOISE_FRACTION_HIGH):
    print(noise_report(f))

def fit_case(model, truth, theta_true, layout, sigma, alpha, seed, times=OBS_TIMES):
    # synthetic readings of the TRUE total field; fit by `model`; evaluate against the truth tags afterwards
    tr = cache.get(truth, solver)
    obs = observe(tr.field(theta_true), layout, ObservationSpec(times, sigma, seed))
    fit = fit_sources(model, obs, InferenceOptions(alpha=alpha, solver=solver), cache)
    mr = cache.get(model, solver)
    est = contributions(mr, fit.q_hat, zero_parts(mr))
    tc = contributions(tr, theta_true, zero_parts(tr))
    ce = component_errors(est, tc, model, T_EXCL)
    return {"fit": fit, "est": est, "true": tc, "ce": ce, "ident": identifiability_report(fit), "obs": obs}

def ident_table(res, truth_values):
    rows = []
    for p, tv in zip(res["ident"]["params"], truth_values):
        rows.append({"parameter": p["name"], "units": p["units"], "true": tv, "estimate": p["q_hat"], "q_std": p["q_std"],
                     "q_std/q_scale": p["q_std_over_scale"], "column norm": p["column_norm"]})
    return pd.DataFrame(rows)

res = fit_case(scn, scn, theta, LAYOUTS["interior"], NOISE_FRACTION * SCALE, 0.0, SEED)
print("\nfit converged (optimiser success ONLY, not identifiability):", res["fit"].success)
print(ident_table(res, theta).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

In [ ]:
def amplitude_plot(res, theta_true, title):
    fit = res["fit"]; kinds = np.array(fit.param_kinds); names = np.array(fit.param_names)
    fig, axes = plt.subplots(1, 2, figsize=(8, 3.2), gridspec_kw={"width_ratios": [3, 1.2]})
    for ax, kind, unit in zip(axes, ("emission", "inflow"), ("ug/s", "ug/m3")):
        m = kinds == kind; x = np.arange(m.sum())
        ax.bar(x - 0.18, theta_true[m], 0.36, label="true")
        ax.bar(x + 0.18, fit.q_hat[m], 0.36, yerr=fit.q_std[m], capsize=3, label="estimate +- q_std")
        ax.set_xticks(x); ax.set_xticklabels(names[m]); ax.set_ylabel(unit); ax.set_title(f"{kind} amplitudes ({unit})", fontsize=9)
    axes[0].legend(fontsize=8); fig.suptitle(title, y=1.04); plt.show()

amplitude_plot(res, theta, f"true vs estimated amplitudes, noise {100 * NOISE_FRACTION:.0f} % of the signal scale")

In [ ]:
def compare_contrib(res, t=T_END):
    rows = [("local", res["true"]["local"], res["est"]["local"]),
            ("external (sources + boundary)", external_part(res["true"]), external_part(res["est"])),
            ("total", total_field(res["true"]), total_field(res["est"]))]
    fig, axes = plt.subplots(len(rows), 3, figsize=(10, 3.0 * len(rows)))
    d = scn.domain
    for i, (nm, tru, est) in enumerate(rows):
        vmax = max(float(tru.sel(time=t).max()), float(est.sel(time=t).max()))
        diff = (est - tru).sel(time=t).values; dmax = max(float(abs(diff).max()), 1e-12)
        for j, (a, vm, cm, tt) in enumerate(((tru.sel(time=t).values, vmax, "viridis", "true"), (est.sel(time=t).values, vmax, "viridis", "estimated"),
                                              (diff, dmax, "RdBu_r", "estimated - true"))):
            im = axes[i][j].imshow(a, origin="lower", extent=(0, d.lx, 0, d.ly), vmin=-vm if j == 2 else 0, vmax=vm, cmap=cm)
            axes[i][j].add_patch(Rectangle((scn.study.x0, scn.study.y0), STUDY, STUDY, fill=False, ec="k", lw=1))
            axes[i][j].set_title(f"{nm}: {tt}", fontsize=8); axes[i][j].set_xticks([]); axes[i][j].set_yticks([])
            fig.colorbar(im, ax=axes[i][j], shrink=0.75)
    plt.tight_layout(); plt.show()

compare_contrib(res)
ce = res["ce"]
print({k: (None if v is None else round(v, 5)) for k, v in ce.items()})
print("emission relative L2 errors:", {k: v for k, v in emission_error(res["fit"].q_hat, theta, scn).items() if k != "inflow"},
      "| inflow abs error (ug/m3):", emission_error(res["fit"].q_hat, theta, scn)["inflow"]["abs_error_ug_m3"])

### Identifiability: do not read it off the optimiser

`success=True` only says the optimiser stopped; it does **not** say the data pin the parameters down. Look at:
* **singular values** of the whitened, scaled sensitivity matrix (measured in units of "one q_scale at the stated noise"): a direction with singular value below 1 is constrained *worse than one scale*;
* **rank** (numerical) and **effective rank** (number of singular values above 1);
* **condition number** (largest / smallest singular value);
* **q_std / q_scale**: posterior standard deviation relative to the scale; large means the data say little about this parameter;
* **flags**: `rank_deficient`, `unidentifiable:<name>`, `confounded:<a>,<b>` (thresholds are defined in `pravah.attribution` and described in `docs/transboundary.md`).

In [ ]:
def show_ident(res, title):
    r = res["ident"]
    print(title)
    print("  singular values :", np.round(r["singular_values"], 3))
    print(f"  rank {r['rank']}, effective rank {r['effective_rank']}, condition {r['condition']:.3g}")
    print("  flags           :", r["flags"] if r["flags"] else "none")

show_ident(res, f"mixed scenario, interior layout, {100 * NOISE_FRACTION:.0f} % noise")
res_hi = fit_case(scn, scn, theta, LAYOUTS["interior"], NOISE_FRACTION_HIGH * SCALE, 0.0, SEED)
show_ident(res_hi, f"mixed scenario, interior layout, {100 * NOISE_FRACTION_HIGH:.0f} % noise")
print(ident_table(res_hi, theta).to_string(index=False, float_format=lambda v: f"{v:.4g}"))

Interactive version (optional). Pick a noise level and a seed and press **Run**; nothing is recomputed until you press the button, and the cached responses make each press fast. If `ipywidgets` is missing, the plain call underneath is used instead.

In [ ]:
def interactive_report(frac, alpha, seed):
    r = fit_case(scn, scn, theta, LAYOUTS["interior"], frac * SCALE, alpha, int(seed))
    print(noise_report(frac))
    print("estimates:", dict(zip(r["fit"].param_names, np.round(r["fit"].q_hat, 4))))
    print("true     :", dict(zip(r["fit"].param_names, np.round(theta, 4))))
    show_ident(r, f"alpha = {alpha:g}")
    print("external fraction: true", round(r["ce"]["external_fraction_true"], 4), "estimated", round(r["ce"]["external_fraction_est"], 4))

try:
    import ipywidgets as widgets
    from IPython.display import display
    nw = widgets.FloatSlider(value=NOISE_FRACTION, min=0.0, max=0.5, step=0.01, description="noise frac")
    aw = widgets.FloatLogSlider(value=1.0, base=10, min=-4, max=5, step=1, description="alpha")
    sw = widgets.IntSlider(value=SEED, min=0, max=50, description="seed")
    bw = widgets.Button(description="Run"); ow = widgets.Output()
    def _click(_):
        with ow:
            ow.clear_output(); interactive_report(nw.value, aw.value, sw.value)
    bw.on_click(_click)
    display(widgets.VBox([nw, aw, sw, bw, ow]))
except ImportError:
    print("ipywidgets not installed; plain call with noise fraction 0.05, alpha 1, seed 3 (edit the numbers and re-run).")
    interactive_report(0.05, 1.0, 3)

## 5. Same sensor budget: interior sites only vs including an upwind reference site

Eight sensors either way. `interior8` places all eight inside the study region; `interior+upwind` swaps two of them for sites at the **western edge of the domain, outside the study region**, where they measure air *before* it has crossed the region. Intuition: an upwind sensor sees external (inflow) pollution almost alone, so it separates the external contribution from the local one. We test this rather than assume it: the same noise level (the stated signal scale is the *same* number for both layouts) and the same noise seeds, `N_REPS` repetitions. Errors are relative to the peak study-region total.

Caveat: this is **one** parameter set and one wind, not an ensemble. The production run (`configs/transboundary.json`) uses validation/test ensembles with paired layout comparisons.

In [ ]:
rows = []
for lname in ("interior8", "interior+upwind"):
    for frac in (NOISE_FRACTION, NOISE_FRACTION_HIGH):
        for rep in range(N_REPS):
            r = fit_case(scn, scn, theta, LAYOUTS[lname], frac * SCALE, 0.0, 100 + rep)
            ce_, idr = r["ce"], r["ident"]
            rows.append({"layout": lname, "noise": frac, "rep": rep,
                         "rmse_total/peak": ce_["rmse_total"] / peak_study, "rmse_external/peak": ce_["rmse_external"] / peak_study,
                         "|F_ext error|": abs(ce_["external_fraction_error"]), "E1 rel err": abs(r["fit"].q_hat[2] - theta[2]) / theta[2],
                         "uniform abs err (ug/m3)": abs(r["fit"].q_hat[3] - theta[3]),
                         "q_std/scale uniform": idr["params"][3]["q_std_over_scale"], "condition": idr["condition"],
                         "flagged": bool(idr["flags"])})
lay_df = pd.DataFrame(rows)
summ = lay_df.groupby(["noise", "layout"]).agg(["mean", "std"]).drop(columns=["rep", "flagged"], level=0, errors="ignore")
print(summ.round(4).to_string())
print("\nfits with any identifiability flag:", int(lay_df["flagged"].sum()), "of", len(lay_df))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.3))
for ax, col in zip(axes, ("rmse_external/peak", "|F_ext error|", "uniform abs err (ug/m3)")):
    for lname, mk in (("interior8", "o"), ("interior+upwind", "s")):
        g = lay_df[lay_df["layout"] == lname].groupby("noise")[col]
        ax.errorbar(g.mean().index, g.mean().values, yerr=g.std().values, marker=mk, capsize=3, label=lname)
    ax.set_xlabel("noise fraction"); ax.set_title(col, fontsize=9)
axes[0].legend(fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
mean = lay_df.drop(columns=["rep", "flagged"]).groupby(["noise", "layout"]).mean()
ratio = (mean.xs("interior+upwind", level="layout") / mean.xs("interior8", level="layout")).drop(columns="condition")
print("ratio of means, interior+upwind / interior8 (below 1 = the upwind layout is better):")
print(ratio.round(2).to_string())

Read the table as the data say, not as we hoped: compare means **and** spread across noise draws; a difference that is not clearly larger than the spread is not evidence. With the default settings the upwind layout pins down the **boundary amplitude** much better (its `q_std/scale uniform` and absolute error are far smaller: an upwind site sees the inflow before local sources mix with it), but it does **not** improve the total-field error or the external fraction here, because two interior sensors were given up to buy the two upwind ones. A reference site is therefore a trade, not a free gain; which layout is better depends on what you need to estimate.

## 6. Hard cases: when the total fits but the decomposition does not

### 6a. Confounded sources
`E1c` is an external source placed on the **wind line through L1**, upwind of it. Their downwind concentration patterns are then almost the same shape, shifted in time, so the sensors can hardly tell "more L1" from "more E1c". We use a notebook-local override (a faster 8 m/s wind and `T` = 14400 s, as in the test suite) so that the two columns become nearly collinear.

In [ ]:
import copy
cfg_c = copy.deepcopy(cfg)
cfg_c["physics"].update({"T": 14400.0, "dt": 60.0})
cfg_c["scenarios"]["confounded"]["wind_speed"] = 8.0
cscn, ctheta, clab = tb.build_scenario(cfg_c, "confounded")
CT = obs_times(14400.0)
CLAY = to_layout(load_config(root / "configs" / "transboundary.json")["special"]["confounded_layout"], "C")
cresp = cache.get(cscn, solver)
c_scale = signal_scale(cresp, ctheta, CLAY, CT)
print("confounded layout signal scale:", round(c_scale, 4), "ug/m3; E1c position (study-relative m):",
      (round(clab[2]["x_study"]), round(clab[2]["y_study"])))
for frac in (NOISE_FRACTION, 0.30):
    r = fit_case(cscn, cscn, ctheta, CLAY, frac * c_scale, 0.0, 7, CT)
    cos = r["fit"].column_cosines[0, 2]
    show_ident(r, f"confounded, noise {100 * frac:.0f} % of scale ({frac * c_scale:.4f} ug/m3), |cos(L1, E1c)| = {abs(cos):.4f}")
    print("  q_std/q_scale:", np.round(r["fit"].q_std / r["fit"].q_scale, 3))

# one noise draw can be lucky: repeat over 20 noise seeds and report mean (sd)
conf_rows = []
for frac in (NOISE_FRACTION, NOISE_FRACTION_HIGH, 0.30):
    for rep in range(20):
        r = fit_case(cscn, cscn, ctheta, CLAY, frac * c_scale, 0.0, 500 + rep, CT)
        pk = float(total_field(r["true"]).sel(time=slice(T_EXCL, None)).max())
        q = r["fit"].q_hat
        conf_rows.append({"noise": frac, "total RMSE / peak": r["ce"]["rmse_total"] / pk,
                          "|F_ext error|": abs(r["ce"]["external_fraction_error"]),
                          "L1 rel. error": abs(q[0] - ctheta[0]) / ctheta[0], "E1c rel. error": abs(q[2] - ctheta[2]) / ctheta[2],
                          "flags": ";".join(r["ident"]["flags"])})
cdf = pd.DataFrame(conf_rows)
print("\nmean (sd) over 20 noise draws:")
print(cdf.drop(columns="flags").groupby("noise").agg(["mean", "std"]).round(4).to_string())
print("\ndraws with the confounded flag:", int(cdf["flags"].str.contains("confounded").sum()), "of", len(cdf),
      "| draws with rank_deficient:", int(cdf["flags"].str.contains("rank_deficient").sum()))

Compare the **total** error (relative to the peak) with the errors of the individual **amplitudes** and of `F_ext` in the multi-seed table: the total field is reproduced far better than its split into origins, (the amplitudes are off by a much larger relative amount than the total). *A good fit of the total is not evidence that the decomposition is right.* The `confounded:L1,E1c` flag (and, at higher noise, effective rank below 3 and `q_std/q_scale` near or above 1 for the two confounded sources) is the warning that the data cannot separate them. (A single noise draw can be lucky, which is why the table repeats the fit.)

### 6b. No arrival: the external source never reaches the sensors
Here the wind blows away from the region. We **verify with the model** that the external source does not reach the study region (rather than assuming it), then fit anyway.

In [ ]:
nscn, ntheta, _ = tb.build_scenario(cfg, "no_arrival")
nresp = cache.get(nscn, solver)
nc = contributions(nresp, ntheta, zero_parts(nresp))
ratio = tb.verify_no_arrival(nc, nscn, float(cfg["special"]["no_arrival_max_ratio"]))
print(f"model check: (study-mean external)/(study-mean local) = {ratio:.3g} (must be < {cfg['special']['no_arrival_max_ratio']:g})")
for a in (0.0, ALPHA_MODERATE):
    r = fit_case(nscn, nscn, ntheta, LAYOUTS["interior"], NOISE_FRACTION * SCALE, a, 11)
    print(f"\nalpha = {a:g}: optimiser success = {r['fit'].success}")
    print(ident_table(r, ntheta).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
    print("  flags:", r["ident"]["flags"], "| singular values:", np.array2string(np.array(r["ident"]["singular_values"]), precision=3))

The `E1` column has (numerically) zero norm: the data carry **no information** about it. With `alpha = 0` its estimate is an arbitrary, enormous number (the optimiser still reports `success`!); with a prior (`alpha > 0`, prior mean 0) the estimate is simply the prior, numerically **0**, not the true amplitude. Neither is a recovery. `q_std/q_scale` is astronomically large without a prior and equals 1 (the prior width) with one, and the flags say `unidentifiable:E1`. The honest statement is "the sensors cannot tell us anything about E1 under this wind".

### 6c. Wind mismatch
The truth has a different wind than the one the inversion assumes: rotated by `rotation_deg` and with speed error `speed_error` (both from the config). Nothing in the identifiability diagnostics is told about this.

In [ ]:
wm = cfg["special"]["wind_mismatch"]
rows = []
for tag, rot, spd in (("matched wind", 0.0, 0.0), ("mismatched wind", float(wm["rotation_deg"]), float(wm["speed_error"]))):
    truth = perturb_wind(scn, spd, rot)
    r = fit_case(scn, truth, theta, LAYOUTS["interior"], NOISE_FRACTION * SCALE, 0.0, 21)
    rows.append({"case": tag, "estimates (L1, L2, E1, uniform)": np.round(r["fit"].q_hat, 3).tolist(),
                 "total RMSE/peak": r["ce"]["rmse_total"] / peak_study, "external RMSE/peak": r["ce"]["rmse_external"] / peak_study,
                 "F_ext error": r["ce"]["external_fraction_error"], "flags": ";".join(r["ident"]["flags"]) or "none"})
print(pd.DataFrame(rows).round(4).to_string(index=False))

With a mismatched wind the amplitudes and errors change while the identifiability flags need not: the flags only describe the *assumed* model's sensitivity matrix, so they **cannot detect a wrong model**. Model error has to be checked separately (e.g. on held-out sensors).

### 6d. Prior sensitivity
Ridge strength `alpha` pulls the estimate towards the prior (zero here), in scaled units. `alpha = 0` is **no prior**; a small value is a weak prior; `ALPHA_STRONG` is a strong one. We show how much the answer depends on it. These values are fixed in advance; in the research run `alpha` is chosen on separate *validation* cases and frozen before the test ensemble.

In [ ]:
rows = []
for nm, (sc, th_, lay, tt, scl) in {"mixed": (scn, theta, LAYOUTS["interior"], OBS_TIMES, SCALE),
                                    "confounded (30 % noise)": (cscn, ctheta, CLAY, CT, c_scale)}.items():
    frac = 0.30 if "confounded" in nm else NOISE_FRACTION_HIGH
    for a in (0.0, 1e-2, ALPHA_MODERATE, ALPHA_STRONG):
        r = fit_case(sc, sc, th_, lay, frac * scl, a, 31, tt)
        pk = float(total_field(r["true"]).sel(time=slice(T_EXCL, None)).max())
        rows.append({"scenario": nm, "noise": frac, "alpha": a, "estimate": np.array2string(r["fit"].q_hat, precision=3, suppress_small=True),
                     "total RMSE/peak": r["ce"]["rmse_total"] / pk, "external RMSE/peak": r["ce"]["rmse_external"] / pk,
                     "F_ext est": r["ce"]["external_fraction_est"], "F_ext true": r["ce"]["external_fraction_true"]})
print(pd.DataFrame(rows).round(4).to_string(index=False))

A strong prior makes the total fit poor and shrinks everything towards zero, i.e. the answer is then the prior, not the data. Weak or no prior leaves the data in charge but lets poorly constrained directions (confounded, unidentifiable) wander. Always report which one you used.

### 6e. What can the observations distinguish?
* **Can** distinguish origins when their *space-time signatures* differ: the explicit external plume arrives late and from a direction different from local plumes; an upwind sensor sees external pollution before local sources mix with it.
* **Cannot** distinguish sources whose responses at the sensors are nearly proportional (collinear): same direction and similar delay, as in 6a. Then only a **combination** is identified, not the split.
* **Cannot** say anything about a source whose plume never reaches any sensor (6b); the estimate is then the prior or an arbitrary number.
* **Cannot** detect a wrong wind or a wrong source geometry from the identifiability diagnostics (6c).
* A well-fitted total is **necessary, not sufficient** for a correct decomposition.

## 7. Literal indoor box (separate from the outdoor inverse problem)

A building exchanges air with the outside. We model it as one well-mixed box:

`dc_in/dt = P * a * c_out(t) - (a + k_dep) * c_in + q_in(t) / V`

| symbol | meaning | unit |
|---|---|---|
| `c_in`, `c_out` | indoor and outdoor concentration | µg/m³ |
| `P` | penetration factor of outdoor air through the shell | dimensionless, 0 to 1 |
| `a` | air-exchange rate | 1/s |
| `k_dep` | indoor deposition / removal rate | 1/s |
| `V` | indoor volume | m³ |
| `q_in` | indoor emission rate | µg/s |

**One-way coupling.** The box *reads* the outdoor field at the building. Indoor exhaust is **not** added to any outdoor source, so nothing here changes the outdoor field or the outdoor inverse problem above. This section is a separate toy, not part of the municipality-wide outdoor sensor-placement objective.

First, two exactness checks of the integrator against closed forms.

In [ ]:
box = indoor.IndoorBox("house", 6500.0 + BUF, 5000.0 + BUF, P=0.8, a=1.0 / 3600.0, k_dep=0.5 / 3600.0, V=300.0)
print(f"building at ({box.x:.0f}, {box.y:.0f}) m (simulation coordinates); P={box.P}, a={box.a:.3g} 1/s, k_dep={box.k_dep:.3g} 1/s, V={box.V:.0f} m3  (demonstration values)")

# (1) constant forcing: c_out = C0, q_in = Q0 constant, c_in(0) = 0  ->  c_in = c_ss (1 - exp(-lam t))
tt = np.linspace(0.0, 4 * 3600.0, 49)
C0, Q0 = 0.05, 3e-3
num = indoor.indoor_response(box, tt, np.full_like(tt, C0), ((0.0, Q0),))
c_ss = (box.P * box.a * C0 + Q0 / box.V) / box.lam
ana = c_ss * (1.0 - np.exp(-box.lam * tt))
print(f"constant forcing: steady state {c_ss:.5f} ug/m3; max |numerical - analytical| = {np.abs(num - ana).max():.3e} ug/m3")

# (2) limiting case lam = 0 (no exchange, no loss): pure accumulation q t / V
box0 = indoor.IndoorBox("sealed", 0.0, 0.0, P=0.8, a=0.0, k_dep=0.0, V=300.0)
acc = indoor.indoor_response(box0, tt, np.full_like(tt, C0), ((0.0, Q0),))
print(f"lam = 0: max |numerical - q t / V| = {np.abs(acc - Q0 * tt / box0.V).max():.3e} ug/m3")

### Paired indoor / outdoor readings from a building placed in the simulated field
We place the building in the **mixed** outdoor field computed above. A pair of sensors records the outdoor concentration right at the building (`y_out`) and the indoor concentration (`y_in`), both with noise. The building parameters are **known**; the only unknown is one indoor emission amplitude `q_in` (µg/s, constant, non-negative).

In [ ]:
field = resp_mixed.field(theta)
ts = np.asarray(field["time"].values, dtype=float)
t_out, c_out_true = indoor.outdoor_at_building(field, box)
Q_TRUE = 3e-3   # ug/s, demonstration value
sig_out = NOISE_FRACTION_HIGH * float(c_out_true.max())
sig_in = sig_out
print(f"outdoor concentration at the building: peak {c_out_true.max():.4f} ug/m3; indoor/outdoor noise std {sig_out:.5f} ug/m3 "
      f"({100 * NOISE_FRACTION_HIGH:.0f} % of that peak)")
pair = indoor.paired_observations(box, field, ts, Q_TRUE, sig_in, sig_out, seed=5)

# analyst: indoor-driven part computed from the OUTDOOR READINGS y_out (the paired outdoor sensor), then fit q_in
driven_from_reading = indoor.indoor_response(box, ts, pair["y_out"], ((0.0, 0.0),))
fit_in = indoor.fit_indoor_emission(box, pair["y_in"], sig_in, driven_from_reading, pair["g_in"])
print(f"estimate q_in = {fit_in.q_hat:.3e} ug/s (true {Q_TRUE:.1e}); std {fit_in.std:.2e}; singular value {fit_in.singular_value:.3g}; identifiable: {fit_in.identifiable}")

# 100 noise draws: spread of the estimate
est = []
for s in range(100):
    p = indoor.paired_observations(box, field, ts, Q_TRUE, sig_in, sig_out, seed=1000 + s)
    d = indoor.indoor_response(box, ts, p["y_out"], ((0.0, 0.0),))
    est.append(indoor.fit_indoor_emission(box, p["y_in"], sig_in, d, p["g_in"]).q_hat)
print(f"100 noise draws: mean {np.mean(est):.3e}, sd {np.std(est, ddof=1):.2e} ug/s (true {Q_TRUE:.1e})")

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.plot(ts, c_out_true, label="outdoor (true)"); ax.plot(ts, pair["c_in_true"], label="indoor (true)")
ax.plot(ts, pair["driven"], "--", label="indoor from outdoor only")
ax.plot(ts, pair["y_out"], ".", c="C0", alpha=0.5, label="y_out"); ax.plot(ts, pair["y_in"], ".", c="C1", alpha=0.5, label="y_in")
ax.set_xlabel("time (s)"); ax.set_ylabel("ug/m3"); ax.legend(fontsize=7); plt.show()

### What happens if indoor emission is ignored
An analyst who reads the indoor sensor as if all of it came from outdoor air would divide by the steady-state indoor/outdoor ratio `P a / (a + k_dep)`. Any indoor emission then shows up as extra *outdoor* pollution:

In [ ]:
mis = indoor.misattribution_demo(box, field, ts, Q_TRUE)
k = ts >= T_EXCL
print(f"true outdoor peak            : {mis['c_out_true'].max():.4f} ug/m3")
print(f"naive 'outdoor equivalent'    : peak {mis['outdoor_equivalent'].max():.4f} ug/m3")
print(f"overstatement (from indoor q) : mean {mis['overstatement'][k].mean():.4f}, max {mis['overstatement'].max():.4f} ug/m3, "
      f"i.e. {100 * mis['overstatement'][k].mean() / mis['c_out_true'][k].mean():.0f} % of the mean true outdoor value (t >= t_exclude)")
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(ts, mis["c_out_true"], label="true outdoor"); ax.plot(ts, mis["outdoor_equivalent"], label="naive outdoor-equivalent from indoor reading")
ax.set_xlabel("time (s)"); ax.set_ylabel("ug/m3"); ax.legend(fontsize=8); plt.show()

### Indoor-only sensing is ambiguous; a paired design is not
With only an indoor sensor, a **constant** outdoor level and a constant indoor emission produce proportional indoor signals (cosine 1): the sensor cannot tell "outdoors is polluted" from "someone is cooking". Adding an outdoor sensor at the building adds a row that only the outdoor amplitude touches, which breaks the tie.

In [ ]:
sg = sig_in
amb_const = indoor.indoor_only_ambiguity(box, ts, sigma=sg, sigma_out=sig_out)
amb_real = indoor.indoor_only_ambiguity(box, ts, sigma=sg, shape=c_out_true / c_out_true.max(), sigma_out=sig_out)
for nm, a in (("constant outdoor level", amb_const), ("time-varying outdoor level (from the field)", amb_real)):
    print(f"{nm}: cosine between indoor columns = {a.cosine:.6f}")
    print(f"   singular values, indoor-only : {np.array2string(a.sv_indoor_only, precision=4)}")
    print(f"   singular values, paired      : {np.array2string(a.sv_paired, precision=4)}")

With the constant outdoor level the indoor-only design has a (numerically) zero second singular value: one combination is unconstrained. The paired design lifts it. A time-varying outdoor signal helps indoor-only sensing a little, but the columns stay highly correlated, so the second singular value remains small relative to the first.

**Scope.** This indoor box is a single-building toy with *known* building parameters and *one* unknown. It is separate from the outdoor inverse problem and is **not** part of the municipality-wide outdoor sensor-placement objective.

## 8. What this notebook does not show
* No real measurements: the "truth" is generated by the same model that is fitted (an *inverse crime* in the strict sense); real data would add model error far beyond the wind-mismatch test.
* One parameter set and wind in most sections; ensembles, validation-selected regularisation and test-set statistics are in the research profile (`docs/transboundary.md`).
* No identification of any real foreign or distant emitter, and no statement about any real border.

In [ ]:
print(f"notebook wall time: {time.perf_counter() - T_START:.0f} s")